# Phase 4 — LLM Extraction & Three-Model Comparison

In [ ]:
# ── LLM Configuration ──────────────────────────────────────────────────────
# Model is set via GEMINI_MODEL in ../.env
# Supported: gemini-3.1-flash-image-preview | gemini-3-pro-image-preview
# ───────────────────────────────────────────────────────────────────────────

In [ ]:
import numpy as np
import pandas as pd
import os
import json
import time
import gc
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score, confusion_matrix
from sklearn.calibration import calibration_curve
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns

from dotenv import load_dotenv
load_dotenv('../.env')

In [ ]:
from google import genai

ALLOWED_MODELS = {"gemini-3.1-flash-image-preview", "gemini-3-pro-image-preview"}

MODEL_NAME = os.environ.get("GEMINI_MODEL", "gemini-3.1-flash-image-preview")
assert MODEL_NAME in ALLOWED_MODELS, f"GEMINI_MODEL must be one of {ALLOWED_MODELS}, got '{MODEL_NAME}'"

LLM_CLIENT = genai.Client(api_key=os.environ.get("GOOGLE_API_KEY"))

def retry_with_backoff(func, max_retries=5, initial_delay=1.0, multiplier=2.0):
    delay = initial_delay
    for attempt in range(max_retries):
        try:
            return func()
        except Exception as e:
            if attempt == max_retries - 1: raise
            print(f"  Attempt {attempt+1} failed: {e}. Retrying in {delay:.1f}s...")
            time.sleep(delay)
            delay *= multiplier

def extract_text_via_llm(prompt):
    def _call():
        response = LLM_CLIENT.models.generate_content(model=MODEL_NAME, contents=prompt)
        return response.text
    return retry_with_backoff(_call)

print(f"Using Gemini model: {MODEL_NAME}")

## Step 4a — LLM Feature Extraction

In [ ]:
from google.genai import types


def _strip_fences(raw):
    raw = raw.strip()
    if raw.startswith('```'): raw = raw.split('\n', 1)[1]
    if raw.endswith('```'): raw = raw.rsplit('```', 1)[0]
    return raw


def _extract_from_image(sk_id, path, prompt, doc_type):
    """Shared helper for image-based extraction (paystub, linkedin, id)."""
    if not os.path.exists(path): return None
    with open(path, 'rb') as f:
        img_data = f.read()
    contents = [types.Content(parts=[
        types.Part.from_bytes(data=img_data, mime_type="image/png"),
        types.Part.from_text(text=prompt)
    ])]
    def _call():
        response = LLM_CLIENT.models.generate_content(model=MODEL_NAME, contents=contents)
        return response.text
    raw = retry_with_backoff(_call)
    try:
        data = json.loads(_strip_fences(raw))
        data['SK_ID_CURR'] = sk_id
        data['doc_type'] = doc_type
        return data
    except:
        return {'SK_ID_CURR': sk_id, 'doc_type': doc_type, 'parse_error': True}


def _extract_from_pdf(sk_id, path, prompt_body, doc_type):
    """Shared helper for PDF-based extraction (bank_statement, property, credit_report).
    Sends the raw PDF bytes to Gemini so the model reads the rendered document."""
    if not os.path.exists(path): return None
    with open(path, 'rb') as f:
        pdf_data = f.read()
    contents = [types.Content(parts=[
        types.Part.from_bytes(data=pdf_data, mime_type="application/pdf"),
        types.Part.from_text(text=prompt_body)
    ])]
    def _call():
        response = LLM_CLIENT.models.generate_content(model=MODEL_NAME, contents=contents)
        return response.text
    raw = retry_with_backoff(_call)
    try:
        data = json.loads(_strip_fences(raw))
        data['SK_ID_CURR'] = sk_id
        data['doc_type'] = doc_type
        return data
    except:
        return {'SK_ID_CURR': sk_id, 'doc_type': doc_type, 'parse_error': True}


def extract_from_paystub(sk_id):
    return _extract_from_image(sk_id,
        f'../unstructured_data/paystubs/{sk_id}_paystub.png',
        """Analyze this paystub image and extract the following fields as JSON with a confidence score (1-5) per field:
- gross_pay (number)
- net_pay (number)
- net_to_gross_ratio (number, net/gross)
- tenure_years (number, from hire date)
- missing_field_indicator (boolean, true if retirement/401k deduction is absent)
- ocr_noise_score (number 0-1, estimated document quality, 1=clean)

Return ONLY valid JSON, no commentary.""", 'paystub')


def extract_from_linkedin(sk_id):
    return _extract_from_image(sk_id,
        f'../unstructured_data/linkedin/{sk_id}_linkedin.png',
        """Analyze this LinkedIn profile screenshot and extract:
- job_hop_count (number of different employers)
- seniority_score (1-6: 1=Entry, 2=Associate, 3=Mid, 4=Senior, 5=Manager/Director, 6=VP+)
- skill_count (number of skills listed)
- career_progression_index (0-1, promotions vs lateral moves)
- connections_estimate (number)

Return ONLY valid JSON with confidence (1-5) per field, no commentary.""", 'linkedin')


def extract_from_bank_statement(sk_id):
    return _extract_from_pdf(sk_id,
        f'../unstructured_data/bank_statements/{sk_id}_bank_statement.pdf',
        """Analyze this bank statement PDF and extract the following fields as JSON with a confidence score (1-5) per field:
- monthly_income (number)
- monthly_repayment (number)
- overdue_flag (boolean)
- credit_sum (total outstanding credit)
- credit_debt (outstanding debt)

Return ONLY valid JSON with confidence (1-5) per field.""", 'bank_statement')


def extract_from_id_document(sk_id):
    return _extract_from_image(sk_id,
        f'../unstructured_data/id_documents/{sk_id}_id.png',
        """Extract from this ID card image:
- gender (M/F)
- date_of_birth (YYYY-MM-DD)
- family_status (string)
- cnt_children (number)

Return ONLY valid JSON with confidence (1-5) per field.""", 'id_document')


def extract_from_property_doc(sk_id):
    return _extract_from_pdf(sk_id,
        f'../unstructured_data/property_docs/{sk_id}_property.pdf',
        """Analyze this property/utility document PDF and extract:
- housing_type (string)
- own_realty (Y/N)
- own_car (Y/N)
- car_age (number, years)
- region_rating (1-3)
- living_area (number, sq ft)
- total_area (number, sq ft)

Return ONLY valid JSON with confidence (1-5) per field.""", 'property_doc')


def extract_from_credit_report(sk_id):
    return _extract_from_pdf(sk_id,
        f'../unstructured_data/credit_reports/{sk_id}_credit_report.pdf',
        """Analyze this credit bureau report PDF and extract:
- ext_source_1 (0-1, reverse-scale from 300-850 credit score)
- ext_source_2 (0-1, reverse-scale from 300-850)
- ext_source_3 (0-1, reverse-scale from 300-850)
- bureau_inquiries_hour (number)
- bureau_inquiries_day (number)
- bureau_inquiries_week (number)
- bureau_inquiries_month (number)
- bureau_inquiries_quarter (number)
- bureau_inquiries_year (number)

Return ONLY valid JSON with confidence (1-5) per field.""", 'credit_report')

In [ ]:
app_train = pd.read_csv('../datasets/application_train.csv')
sample_ids = app_train.head(10)['SK_ID_CURR'].tolist()

EXTRACTORS = {
    'paystub': extract_from_paystub,
    'linkedin': extract_from_linkedin,
    'bank_statement': extract_from_bank_statement,
    'id_document': extract_from_id_document,
    'property_doc': extract_from_property_doc,
    'credit_report': extract_from_credit_report,
}

extraction_results = []
parse_success = 0
parse_fail = 0

for sk_id in sample_ids:
    print(f"Extracting for SK_ID_CURR={sk_id}...")
    for doc_type, extractor in EXTRACTORS.items():
        result = extractor(sk_id)
        if result is not None:
            extraction_results.append(result)
            if result.get('parse_error'):
                parse_fail += 1
            else:
                parse_success += 1

log_path = '../unstructured_data/extraction_results.jsonl'
with open(log_path, 'w') as f:
    for r in extraction_results:
        f.write(json.dumps(r, default=str) + '\n')

total = parse_success + parse_fail
print(f"\nExtraction complete: {parse_success}/{total} valid JSON ({parse_success/total*100:.1f}%)")
print(f"Target: >= 90%")
print(f"Results saved to {log_path}")

## Step 4b — Dataset Reconstruction

In [ ]:
trimmed = pd.read_csv('../datasets/application_train_trimmed.csv')
trimmed_sample = trimmed[trimmed['SK_ID_CURR'].isin(sample_ids)].copy()

extracted_df = pd.DataFrame(extraction_results)
extracted_wide = extracted_df.pivot_table(index='SK_ID_CURR', columns='doc_type', aggfunc='first')
extracted_wide.columns = ['_'.join(col).strip() for col in extracted_wide.columns.values]
extracted_wide = extracted_wide.reset_index()

reconstructed = trimmed_sample.merge(extracted_wide, on='SK_ID_CURR', how='left')
reconstructed.to_csv('../datasets/application_train_reconstructed.csv', index=False)
print(f"Reconstructed dataset shape: {reconstructed.shape}")
print(f"Saved to ../datasets/application_train_reconstructed.csv")

## Step 4c — Three-Model Comparison

In [ ]:
def train_model(features, labels, n_folds=5):
    """Train LightGBM with 5-fold CV. Returns out-of-fold predictions, metrics, and feature importances."""
    feature_names = list(features.columns)
    X = np.array(features)
    
    k_fold = KFold(n_splits=n_folds, shuffle=True, random_state=50)
    feature_importance_values = np.zeros(len(feature_names))
    out_of_fold = np.zeros(X.shape[0])
    
    valid_scores = []
    train_scores = []
    
    for train_idx, valid_idx in k_fold.split(X):
        train_X, train_y = X[train_idx], labels.iloc[train_idx]
        valid_X, valid_y = X[valid_idx], labels.iloc[valid_idx]
        
        clf = lgb.LGBMClassifier(
            n_estimators=10000, objective='binary',
            class_weight='balanced', learning_rate=0.05,
            reg_alpha=0.1, reg_lambda=0.1,
            subsample=0.8, n_jobs=-1, random_state=50
        )
        
        clf.fit(train_X, train_y, eval_metric='auc',
                eval_set=[(valid_X, valid_y), (train_X, train_y)],
                eval_names=['valid', 'train'],
                callbacks=[lgb.early_stopping(100), lgb.log_evaluation(200)])
        
        best_iter = clf.best_iteration_
        feature_importance_values += clf.feature_importances_ / k_fold.n_splits
        out_of_fold[valid_idx] = clf.predict_proba(valid_X, num_iteration=best_iter)[:, 1]
        
        valid_scores.append(clf.best_score_['valid']['auc'])
        train_scores.append(clf.best_score_['train']['auc'])
        
        gc.enable()
        del clf, train_X, valid_X
        gc.collect()
    
    valid_auc = roc_auc_score(labels, out_of_fold)
    valid_scores.append(valid_auc)
    train_scores.append(np.mean(train_scores))
    
    fold_names = list(range(n_folds)) + ['overall']
    metrics = pd.DataFrame({'fold': fold_names, 'train': train_scores, 'valid': valid_scores})
    fi = pd.DataFrame({'feature': feature_names, 'importance': feature_importance_values})
    
    return out_of_fold, metrics, fi

print("Model training function defined.")

In [ ]:
model_a_auc = float(open('../artifacts/model_a_auc.txt').read().strip())
print(f"Model A benchmark AUC: {model_a_auc:.6f}")

In [ ]:
print("NOTE: Models B and C are trained on the sample subset.")
print("With only 10 test rows, cross-validation results will be indicative only.")
print("Scale to 2,000 rows for production-quality comparison.")

## Evaluation Metrics

In [ ]:
def compute_ks_statistic(y_true, y_pred):
    """Compute Kolmogorov-Smirnov statistic."""
    from scipy.stats import ks_2samp
    pos_scores = y_pred[y_true == 1]
    neg_scores = y_pred[y_true == 0]
    if len(pos_scores) == 0 or len(neg_scores) == 0:
        return 0.0
    stat, _ = ks_2samp(pos_scores, neg_scores)
    return stat

def display_metrics(model_name, auc_val, ks_val=None, gini_val=None):
    """Display metrics for a model."""
    print(f"\n{'='*40}")
    print(f"  {model_name}")
    print(f"{'='*40}")
    print(f"  ROC-AUC:       {auc_val:.6f}")
    if ks_val is not None:
        print(f"  KS Statistic:  {ks_val:.6f}")
    if gini_val is not None:
        print(f"  Gini Coeff:    {gini_val:.6f}")

model_a_gini = 2 * model_a_auc - 1
display_metrics("Model A (Structured Only)", model_a_auc, gini_val=model_a_gini)

print("\n\nModel B and C training requires the sample to have enough rows.")
print("With the current 10-row test sample, full CV is not meaningful.")
print("Scale USE_TEST_SAMPLE=False in notebook 02 and re-run for production results.")

## Success Criteria Validation

In [ ]:
print("=" * 60)
print("SUCCESS CRITERIA CHECK")
print("=" * 60)

criteria = {
    "Document generation success rate >= 95%": None,
    "LLM extraction valid JSON parse rate >= 90%": f"{parse_success}/{total} ({parse_success/total*100:.1f}%)" if total > 0 else "N/A",
    "Extraction accuracy — categorical >= 85%": "Requires production run",
    "Extraction accuracy — numeric >= 80% (±10%)": "Requires production run",
    "ROC-AUC delta (A vs C) < 0.02": "Requires production run",
    "Model B ROC-AUC > 0.70": "Requires production run",
}

gen_log_path = '../unstructured_data/generation_log.jsonl'
if os.path.exists(gen_log_path):
    gen_logs = [json.loads(line) for line in open(gen_log_path)]
    gen_success = sum(1 for l in gen_logs if l.get('success'))
    gen_total = len(gen_logs)
    gen_rate = gen_success / gen_total * 100 if gen_total > 0 else 0
    criteria["Document generation success rate >= 95%"] = f"{gen_success}/{gen_total} ({gen_rate:.1f}%)"

for criterion, result in criteria.items():
    status = result if result else "Not yet evaluated"
    print(f"  {criterion}")
    print(f"    → {status}")
    print()

print("Run with USE_TEST_SAMPLE=False and full pipeline for definitive results.")